This produces reduced.csv (filtered dataset) and creates exploratory and explanatory figures for midterm report

In [1]:
# Load data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data = pd.read_csv("../data/chess_v2_model_versions.csv")

In [ ]:
## Clean Data - makes new file called reduced.csv

# Add 'developer' column to replace mutiple other columns
data["developer"] = data["openrouter__canonical_slug"].str.split("/").str[0]

# Mostly Empty
missing = data.isna().mean()
near_empty = missing[missing > 0.8].index.tolist()

# Constant
constant = data.columns[data.nunique() <= 1].tolist()

# High Cardinality
protect = ["version__Version release date", "openrouter__benchmarks__design_arena"]   
text = data.select_dtypes(exclude="number")
high_card = [c for c in text.columns if text[c].nunique() >= 50 and c not in protect]

# IDs or other non-useful text 
keywords = ["record_id", "link", "log", "notes", "description", "abstract", "reference", "authors", "_id", "last modified"]
id_like = [c for c in data.columns
           if any(k in c.lower() for k in keywords)]
id_like += ["base_model__Organization", "base_model__Task", "base_model__Domain", "openrouter__canonical_slug", "openrouter__supported_parameters"]

# Not available when deploying
unavailable = [c for c in data.columns if c.startswith("run__")] # build_chess_dataset.py adds prefix for info that comes from run

# Redundant
redundant = ["base_model__Publication date", "openrouter__created", "openrouter__created_utc", # release date
             "reasoning_level_from_name", "reasoning_level_from_slug", # combined into reasoning_level
             "version_name_qualifier", # raw text behind reasoning_level
             "openrouter__top_provider__context_length", # context_length
             "openrouter__architecture__input_modalities", # same info as modality
             "openrouter__architecture__tokenizer",      # ≈ developer
             "openrouter__pricing__input_cache_write", "openrouter__pricing__input_cache_write_1h"]  # ≈ prompt price

# Specific Reasons
specific = ["base_model__Confidence", # Epoch confidence on model metadata
            "base_model__Notability criteria", # Epoch's reason for testing model
            "openrouter_match_method", # From building dataset
            "base_model__Reasoning/thinking settings", # 91% "Unknown"
            "openrouter__pricing__overrides", # Nested rules, also 73% missing
            "openrouter__reasoning__supported_efforts", "openrouter__reasoning__default_effort", "openrouter__reasoning__default_enabled" # Model options or defaults, not actually what is used
]


# Drop unused data
to_drop = set(unavailable) | set(near_empty) | set(constant) | set(id_like) | set(high_card) | set(redundant) | set(specific)
reduced = data.drop(columns=to_drop)

reduced.to_csv("../data/reduced.csv", index=False)

In [10]:
# Define Feature types manually
continuous = ["openrouter__benchmarks__artificial_analysis__intelligence_index",
    "openrouter__benchmarks__artificial_analysis__coding_index",
    "openrouter__benchmarks__artificial_analysis__agentic_index",
    "openrouter__context_length",
    "openrouter__top_provider__max_completion_tokens",
    "openrouter__pricing__prompt",
    "openrouter__pricing__completion",
    "openrouter__pricing__input_cache_read",
    "openrouter__pricing__web_search"]

dates = ["version__Version release date",
         "openrouter__knowledge_cutoff"
]

categorical = ["base_model__Model accessibility",
    "base_model__Country (from Organization)",
    "developer",
    "openrouter__architecture__modality",
    "openrouter__top_provider__is_moderated",
    "openrouter__reasoning__mandatory"]

ordinal = ["reasoning_level"]
order = ["none", "minimal", "low", "medium", "high", "xhigh", "max"]

all_listed = continuous + categorical + ordinal + dates
missed = [c for c in reduced.columns if c not in all_listed and c != "score_target"]
missed

['openrouter__benchmarks__design_arena']